In [1]:
import pandas as pd


In [4]:
df=pd.read_csv("output_final (1).csv")
df.head()

,Unnamed: 0,patient_id,note,question,answer,task,llm_response,summary_llm,rouge1,rouge2,rougeL,bertscore_f1,compression_ratio,note_word_count,summary_word_count
0,2,2,Hospital Course Summary: Admission Date: [Inse...,What were the key improvements in the patient'...,"During the hospital course, the patient's medi...",Summarization,"{\n ""patient_id"": ""2"",\n ""demographics"":...",The key improvements in the patient's medical ...,0.6957,0.4602,0.3304,0.9028,15.69,274,43
1,5,5,Discharge Summary: Patient: 52-year-old male h...,How did the patient's treatment for dysphagia ...,"During the patient's hospital stay, treatment ...",Summarization,"{\n ""patient_id"": ""5"",\n ""demographics"":...",The patient's treatment for dysphagia progress...,0.6218,0.3932,0.5210,0.8927,26.24,221,58
2,10,11,Discharge Summary: Patient Name: [REDACTED] Me...,"Can you provide a summary of the treatment, ho...",The 45-year-old female patient with a history ...,Summarization,"{\n ""patient_id"": ""[REDACTED]"",\n ""demog...",A 45-year-old female patient with a history of...,0.5596,0.3246,0.4041,0.8729,24.61,321,79
3,12,13,DISCHARGE SUMMARY: Patient Name: X Medical Rec...,"Based on the given discharge summary, can you ...",The patient with a multifocal invasive mammary...,Summarization,"{\n ""patient_id"": ""X"",\n ""demographics"":...",The patient's treatment plan for managing thei...,0.4578,0.2195,0.3012,0.8435,14.70,347,51
4,18,20,Hospital Course: The patient is a 78-year-old ...,What are the key findings and diagnosis of the...,The key findings of the patient include abnorm...,Summarization,"{\n ""patient_id"": ""20"",\n ""demographics""...",The patient was diagnosed with metastatic Merk...,0.4228,0.3306,0.3740,0.8707,17.05,176,30


In [9]:
import sys
!"{sys.executable}" -m pip install fpdf2

  Using cached defusedxml-0.7.1-py2.py3-none-any.whl.metadata (32 kB)
Using cached defusedxml-0.7.1-py2.py3-none-any.whl (25 kB)

   ---------------------------------------- 0/2 [defusedxml]
   -------------------- ------------------- 1/2 [fpdf2]
   -------------------- ------------------- 1/2 [fpdf2]
   -------------------- ------------------- 1/2 [fpdf2]
   -------------------- ------------------- 1/2 [fpdf2]
   -------------------- ------------------- 1/2 [fpdf2]
   -------------------- ------------------- 1/2 [fpdf2]
   -------------------- ------------------- 1/2 [fpdf2]
   ---------------------------------------- 2/2 [fpdf2]




[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
!"{sys.executable}" -m pip install "fpdf2[fonts]"


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [44]:
import pandas as pd
import json
import random
import os
from fpdf import FPDF

# =============================================================================
# CONFIGURATION
# =============================================================================
# Remplacez cette variable par le chemin réel vers votre image de template
TEMPLATE_IMAGE_PATH = "ministre_logo.png" 

# Liste des médecins
DOCTORS = [
    "Mohamed Amine EL ARBANI", 
    "Yahya SAHNOUN", 
    "Mohamed Nizar MADIH"
]

# =============================================================================
# FONCTIONS UTILITAIRES
# =============================================================================
def get_field(value):
    if not value or value == "Non mentionné" or value == []:
        return "-"
    if isinstance(value, list):
        # On remplace le caractère spécial '•' par un '-' simple (ASCII)
        clean_list = [str(item).strip() for item in value if str(item).strip()]
        if not clean_list:
            return "-"
        return "\n".join([f"- {item}" for item in clean_list])
    return str(value).strip()

# =============================================================================
# CLASSE PDF PERSONNALISÉE
# =============================================================================
class MedicalReportPDF(FPDF):
    def __init__(self):
        super().__init__()
        self.set_font("helvetica", "", 10)

    def add_background_template(self, image_path):
        """Ajoute l'image centrée en haut avec une largeur logique."""
        if os.path.exists(image_path):
            # Définir une largeur logique (ex: 180mm sur une page de 210mm)
            target_width = 80 
            
            # Calculer la position X pour centrer l'image
            # (Largeur_page - largeur_image) / 2
            x_centered = (210 - target_width) / 2
            
            # Ajouter l'image (y=10 pour laisser une marge en haut)
            # FPDF calcule automatiquement la hauteur pour garder les proportions
            self.image(image_path, x=x_centered,y=5, w=target_width)
        else:
            print(f"Attention : Image {image_path} introuvable.")

    def section_title(self, title):
        self.set_font("helvetica", "B", 12)
        self.set_text_color(0, 51, 102)
        self.cell(0, 8, title, ln=True)
        self.line(self.get_x(), self.get_y(), self.get_x() + 190, self.get_y())
        self.ln(3)
        self.set_text_color(0, 0, 0)

    def section_body(self, text):
        self.set_font("helvetica", "", 10)
        # Encodage latin-1 sécurisé
        safe_text = str(text).encode("latin-1", "replace").decode("latin-1")
        self.multi_cell(0, 6, safe_text)
        self.ln(5)

# =============================================================================
# FONCTION PRINCIPALE DE GÉNÉRATION
# =============================================================================
def generate_patient_pdf(df, target_patient_id, output_filename="rapport_medical.pdf"):
    """
    Recherche le patient dans le DataFrame, extrait le JSON et génère le PDF.
    """
    # 1. Recherche du patient
    row = df[df['patient_id'].astype(str) == str(target_patient_id)]
    
    if row.empty:
        print(f"Erreur : Aucun patient trouvé avec l'ID {target_patient_id}")
        return False
    
    # 2. Extraction et parsing du JSON
    try:
        json_str = row.iloc[0]['llm_response']
        data = json.loads(json_str)
    except json.JSONDecodeError:
        print("Erreur : Le contenu de la colonne 'llm_response' n'est pas un JSON valide pour ce patient.")
        return False
    
    # 3. Extraction sécurisée des données
    patient_id = get_field(data.get("patient_id"))
    age = get_field(data.get("demographics", {}).get("age"))
    gender = get_field(data.get("demographics", {}).get("gender"))
    
    clinical = data.get("clinical_data", {})
    analyses = get_field(clinical.get("analyses_performed"))
    past_meds = get_field(clinical.get("past_medications"))
    treatments = get_field(clinical.get("hospital_treatments"))
    future_plan = get_field(clinical.get("future_plan"))
    
    summary = get_field(data.get("summary"))
    
    # Choix aléatoire du médecin
    doctor_name = random.choice(DOCTORS)

    # 4. Construction du PDF
    pdf = MedicalReportPDF()
    pdf.add_page()
    
    # Ajouter le template visuel (l'image fournie)
    pdf.add_background_template(TEMPLATE_IMAGE_PATH)
    
    # Ajustement de la position de départ (Y) pour ne pas écrire sur l'en-tête de l'image
    # (L'en-tête du Ministère prend environ le tiers supérieur, on commence à Y=70mm)
    pdf.set_y(50)
    
    # Titre du document (écrase le "CERTIFICAT MEDICAL" de l'image si on le place au même endroit)
    # Dans ce script, on place notre contenu juste en dessous des logos
    pdf.set_font("helvetica", "BU", 16)
    pdf.cell(0, 10, "RAPPORT MÉDICAL DE SYNTHÈSE", align="C", ln=True)
    pdf.ln(10)
    
    # Informations administratives
    pdf.set_font("helvetica", "B", 11)
    pdf.cell(100, 6, f"Médecin en charge : Dr. {doctor_name}")
    pdf.cell(90, 6, f"Dossier Patient N° : {patient_id}", align="R", ln=True)
    
    pdf.set_font("helvetica", "", 11)
    pdf.cell(100, 6, f"Âge : {age}")
    pdf.cell(90, 6, f"Sexe : {gender}", align="R", ln=True)
    pdf.ln(10)
    
    # Sections du rapport
    pdf.section_title("RÉSUMÉ CLINIQUE")
    pdf.section_body(summary)
    
    pdf.section_title("EXAMENS ET ANALYSES EFFECTUÉS")
    pdf.section_body(analyses)
    
    pdf.section_title("MÉDICAMENTS PRÉCÉDANT L'ADMISSION")
    pdf.section_body(past_meds)
    
    pdf.section_title("SOINS ET TRAITEMENTS HOSPITALIERS")
    pdf.section_body(treatments)
    
    pdf.section_title("PLAN DE SUIVI ET PRESCRIPTIONS À LA SORTIE")
    pdf.section_body(future_plan)
    
    # Zone de signature
    pdf.ln(15)
    pdf.set_y(270)
    pdf.set_font("helvetica", "B", 11)
    pdf.cell(0, 6, "Signature et Cachet du Médecin :", align="R", ln=True)
    
    # 5. Sauvegarde
    pdf.output(output_filename)
    print(f"Succès : Rapport généré avec succès -> {output_filename}")
    return True

# =============================================================================
# EXEMPLE D'UTILISATION (Pour tester le script)
# =============================================================================
if __name__ == "__main__":
    # Simulation d'un DataFrame contenant votre exemple
    exemple_json = """{
        "patient_id": "826",
        "demographics": {
            "age": "10 years old",
            "gender": "Female"
        },
        "clinical_data": {
            "analyses_performed": ["Computed Tomography (CT) scan", "Blood tests", "Urine tests"],
            "past_medications": ["Paracetamol"],
            "hospital_treatments": ["Intravenous antibiotics", "Intrapleural urokinase therapy", "Chest drain insertion"],
            "future_plan": ["Continue prescribed course of antibiotics", "Regular analgesia", "Avoid playing netball", "Follow up within 1-2 weeks"]
        },
        "summary": "The patient was admitted to the hospital due to persistent right-sided flank pain..."
    }"""
    
    df_test = pd.DataFrame({
        'patient_id': ['826'],
        'llm_response': [exemple_json]
    })
    

In [50]:
  # Appel de la fonction
  generate_patient_pdf(df, target_patient_id="20", output_filename="rapport_patient_20.pdf")

Succès : Rapport généré avec succès -> rapport_patient_20.pdf


C:\Users\EL ARBANI\AppData\Local\Temp\ipykernel_19864\2736818189.py:125: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(0, 10, "RAPPORT MÉDICAL DE SYNTHÈSE", align="C", ln=True)
C:\Users\EL ARBANI\AppData\Local\Temp\ipykernel_19864\2736818189.py:131: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(90, 6, f"Dossier Patient N° : {patient_id}", align="R", ln=True)
C:\Users\EL ARBANI\AppData\Local\Temp\ipykernel_19864\2736818189.py:135: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(90, 6, f"Sexe : {gender}", align="R", ln=True)
C:\Users\EL ARBANI\AppData\Local\Temp\ipykernel_19864\2736818189.py:61: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT

True